In [1]:
pip install -q huggingface-hub transformers langchain-dartmouth pypdf chromadb

In [2]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/colab_rag_letters"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, TextLoader
# Notice the underscore here instead of a dot:
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = []
for fname in os.listdir(DATA_DIR):
    path = os.path.join(DATA_DIR, fname)
    if fname.lower().endswith(".pdf"):
        loader = PyPDFLoader(path)
    else:
        loader = TextLoader(path, encoding="utf-8")
    file_docs = loader.load()
    for d in file_docs:
        d.metadata["source_file"] = fname
    docs.extend(file_docs)

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = splitter.split_documents(docs)


In [4]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectordb = Chroma.from_documents(chunks, embedding=emb, persist_directory="/content/chroma_letters")
vectordb.persist()
retriever = vectordb.as_retriever(search_kwargs={"k": 6})


/tmp/ipykernel_13347/2070690054.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/t

In [5]:
def build_query(job_desc: str, resume_text: str) -> str:
    return (
        "Job description:\n" + job_desc +
        "\n\nMy resume for this job:\n" + resume_text +
        "\n\nReturn cover-letter style content matching both."
    )



In [6]:
# Notice the _core here
from langchain_core.prompts import ChatPromptTemplate

template = """
You are an assistant that drafts professional, ATS-friendly cover letters in the user's own style.

Use ONLY the provided context from prior cover letters and project descriptions, plus the job description and resume content.
Do not invent experiences or technologies the user has not used.

Requirements:
- 3–4 paragraphs, 1 page max.
- Match keywords from the job description when they reflect real skills in the resume.
- Maintain a confident but factual tone similar to the context.

Job description:
{job_desc}

Resume content:
{resume_text}

Retrieved context from user's past letters:
{context}

Draft the new cover letter:
"""

prompt = ChatPromptTemplate.from_template(template)


In [8]:
import os
from google.colab import userdata
from langchain_dartmouth.llms import ChatDartmouth
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

os.environ["DARTMOUTH_CHAT_API_KEY"] = userdata.get('dartmouth_api_key')


llm = ChatDartmouth(
    model_name="qwen.qwen3-vl-32b-instruct-fp8",               # Update the model name
    temperature=0.4,
    max_tokens=1000
)

def rag_answer(job_desc, resume_text):
    query = build_query(job_desc, resume_text)
    docs = retriever.invoke(query)
    context = "\n\n".join(d.page_content for d in docs)

    chain_input = {
        "job_desc": job_desc,
        "resume_text": resume_text,
        "context": context,
    }
    full_prompt = prompt.format(**chain_input)
    return llm.invoke(full_prompt).content


In [9]:
job_desc = """Job description goes here"""
resume_text = """Resume text goes here"""

draft = rag_answer(job_desc, resume_text)
print(draft)

I am excited to apply for the position at your organization, bringing with me a strong foundation in technical problem-solving, project documentation, and continuous improvement. Throughout my academic and professional experiences, I have consistently demonstrated a detail-oriented mindset and a commitment to delivering high-quality, data-informed outcomes. Whether supporting computer vision pipelines at Temple Allen Industries or contributing to research initiatives at Two Six Technologies, I have applied structured methodologies to advance technical projects and ensure alignment with team objectives.

My experience includes translating complex technical work into clear, actionable insights—skills that are directly applicable to roles requiring effective communication across technical and non-technical stakeholders. I have also contributed to end-to-end project workflows, from initial planning through execution and documentation, ensuring that processes are both efficient and scalable